# IBGE Municipality Population - Bronze Ingestion

## Imports

In [0]:
import requests
from datetime import datetime
from pyspark.sql.functions import col, current_timestamp, lit, explode
import uuid
import time
from tqdm.notebook import tqdm

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "ibge_municipality_population"
target_table = f"{catalog}.{schema}.{table_name}"
source_system = "ibge"
source_dataset = "municipality_population"

run_id = str(uuid.uuid4())

aggregate_id = "6579"
variable_id = "9324"
years = ["2016", "2017", "2018"]
municipality_level = "N6"
api_base_url = "https://servicodados.ibge.gov.br/api/v3/agregados"

politeness = 1
request_batch = 100
timeout=5

In [0]:
municipalities_df = spark.table(f"{catalog}.bronze.ibge_municipalities")
municipality_ids = municipalities_df.select("ibge_municipality_id").collect()
municipality_ids = [str(row[0]) for row in municipality_ids]
print(len(municipality_ids))

5571


## Call the IBGE API in batches and save response to volume

In [0]:
response_file_paths = []
volume_path = "/Volumes/{catalog}/landing/raw_files/api/{source_dataset}/{response_date}/{table_name}_api_response_{response_date_time}.json"

In [0]:
for i in tqdm(range(0, len(municipality_ids), request_batch)):
    municipality_ids_batch = municipality_ids[i:i + request_batch]
    
    api_url = f"{api_base_url}/{aggregate_id}/periodos/{'|'.join(years)}/variaveis/{variable_id}"
    params = {"localidades": f"{municipality_level}[{','.join(municipality_ids_batch)}]"}

    response = requests.get(
        api_url,
        params=params,
        timeout=timeout
    )
    
    if response.status_code != 200:
        raise RuntimeError(f"Expected 200, got {response.status_code}")
    
    response_date = datetime.now().strftime("%Y%m%d")
    response_date_time = datetime.now().strftime("%Y%m%d_%H%M%S")

    write_file_path = volume_path.format(
        catalog=catalog,
        source_dataset=source_dataset,
        response_date=response_date,
        table_name=table_name,
        response_date_time=response_date_time
    )

    try:
        with open(write_file_path, "w", encoding="utf-8") as f:
            f.write(response.text)

    except FileNotFoundError:
        dbutils.fs.mkdirs("/Volumes/{catalog}/landing/raw_files/api/{source_dataset}/{response_date}/".format(
            catalog=catalog,
            source_dataset=source_dataset,
            response_date=response_date)
        )

        with open(write_file_path, "w", encoding="utf-8") as f:
            f.write(response.text)
    
    response_file_paths.append(write_file_path)
    time.sleep(politeness)


  0%|          | 0/56 [00:00<?, ?it/s]

## Read from volume

In [0]:
source_df = spark.read.json(response_file_paths)

In [0]:
source_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- resultados: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- classificacoes: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- series: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- localidade: struct (nullable = true)
 |    |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |    |-- nivel: struct (nullable = true)
 |    |    |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |    |    |-- nome: string (nullable = true)
 |    |    |    |    |    |-- nome: string (nullable = true)
 |    |    |    |    |-- serie: struct (nullable = true)
 |    |    |    |    |    |-- 2016: string (nullable = true)
 |    |    |    |    |    |-- 2017: string (nullable = true)
 |    |    |    |    |    |-- 2018: string (nullable = true)
 |-- unidade: string (nullable = true)
 

In [0]:
# Explode the top level 'resultados' array
df_res = source_df.withColumn("resultados", explode("resultados"))
df_res.printSchema()

root
 |-- id: string (nullable = true)
 |-- resultados: struct (nullable = true)
 |    |-- classificacoes: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- series: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- localidade: struct (nullable = true)
 |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |-- nivel: struct (nullable = true)
 |    |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |    |-- nome: string (nullable = true)
 |    |    |    |    |-- nome: string (nullable = true)
 |    |    |    |-- serie: struct (nullable = true)
 |    |    |    |    |-- 2016: string (nullable = true)
 |    |    |    |    |-- 2017: string (nullable = true)
 |    |    |    |    |-- 2018: string (nullable = true)
 |-- unidade: string (nullable = true)
 |-- variavel: string (nullable = true)



In [0]:
# Explode the inner 'series' array
df_series = df_res.withColumn("series", explode("resultados.series"))
df_series.printSchema()

root
 |-- id: string (nullable = true)
 |-- resultados: struct (nullable = true)
 |    |-- classificacoes: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- series: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- localidade: struct (nullable = true)
 |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |-- nivel: struct (nullable = true)
 |    |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |    |-- nome: string (nullable = true)
 |    |    |    |    |-- nome: string (nullable = true)
 |    |    |    |-- serie: struct (nullable = true)
 |    |    |    |    |-- 2016: string (nullable = true)
 |    |    |    |    |-- 2017: string (nullable = true)
 |    |    |    |    |-- 2018: string (nullable = true)
 |-- unidade: string (nullable = true)
 |-- variavel: string (nullable = true)
 |-- series: struct (nullable = true)
 |    |-- localidade: struct (nullable =

In [0]:
serie_df = df_series.select(
    col("series.localidade.id").alias("ibge_municipality_id"),
    col("series.localidade.nome").alias("municipality_name"),
    col("series.serie.2016").alias("2016"),
    col("series.serie.2017").alias("2017"),
    col("series.serie.2018").alias("2018"),
    col("_metadata.file_path").alias("source_file_path"),
    col("_metadata.file_modification_time").alias("source_file_modification_time")
)
serie_df.printSchema()

root
 |-- ibge_municipality_id: string (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- 2016: string (nullable = true)
 |-- 2017: string (nullable = true)
 |-- 2018: string (nullable = true)
 |-- source_file_path: string (nullable = false)
 |-- source_file_modification_time: timestamp (nullable = false)



In [0]:
bronze_df = serie_df.unpivot(
    ids=[
        "ibge_municipality_id",
        "municipality_name",
        "source_file_path",
        "source_file_modification_time"
    ],
    values=["2016", "2017", "2018"],
    variableColumnName="year",
    valueColumnName="population"
)

bronze_df.printSchema()

root
 |-- ibge_municipality_id: string (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- source_file_path: string (nullable = false)
 |-- source_file_modification_time: timestamp (nullable = false)
 |-- year: string (nullable = false)
 |-- population: string (nullable = true)



In [0]:
display(bronze_df.limit(5))

ibge_municipality_id,municipality_name,source_file_path,source_file_modification_time,year,population
3156502,Rubelita (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,6937
3156601,Rubim (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,10354
3156700,Sabará (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,135196
3156809,Sabinópolis (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,15961
3156908,Sacramento (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,25819


## Add Bronze metadata

In [0]:
bronze_df = (
    bronze_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

In [0]:
display(bronze_df.limit(5))

ibge_municipality_id,municipality_name,source_file_path,source_file_modification_time,year,population,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
3156502,Rubelita (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,6937,2026-08-12T17:33:50.435Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
3156601,Rubim (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,10354,2026-08-12T17:33:50.435Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
3156700,Sabará (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,135196,2026-08-12T17:33:50.435Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
3156809,Sabinópolis (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,15961,2026-08-12T17:33:50.435Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
3156908,Sacramento (MG),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172743.json,2026-08-12T17:27:43.000Z,2016,25819,2026-08-12T17:33:50.435Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population


## Write to Bronze

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

In [0]:
spark.table(target_table).printSchema()

root
 |-- ibge_municipality_id: string (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- year: string (nullable = true)
 |-- population: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

ibge_municipality_id,municipality_name,source_file_path,source_file_modification_time,year,population,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1100015,Alta Floresta D'Oeste (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,25506,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100023,Ariquemes (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,105896,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100031,Cabixi (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,6289,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100049,Cacoal (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,87877,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100056,Cerejeiras (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,17959,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population


In [0]:
spark.table(target_table).count()

16713